# 02C DeepSeek 实战与失败分析
对应 **L02.07–L02.08**，综合前面工具与循环。

先运行无密钥的失败案例，再按说明开启真实 DeepSeek。默认 `RUN_LIVE=False`，防止“运行全部单元”意外调用收费接口。配置好后由你主动切换为 True。

本课用 DeepSeek 演示真实模型接入。模型列表可能随服务变化，联网前先读取当前账号的 `/models` 列表，再用 `DEEPSEEK_MODEL` 选择实际可用的模型。默认保持离线，避免误发收费请求；离线输出不等于云端联调记录。


## 学习路线与配套课件

实践 2C：模型接入与失败验收（`L02-P03`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件内容：规划与反思、策略选择、上下文、权限、可观测性和任务验收。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, copy
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "02 从大模型到 Agent 组成与最小循环"
sys.path.insert(0, str(CHAPTER / "代码"))
DATA = CHAPTER / "数据/demo-repo"
print("教学资料:", DATA.relative_to(ROOT))
import os, time, urllib.request, urllib.error
from agent_lab import RepoTools, ReplayModel, tool_response, run_agent, TOOLS


教学资料: 02 从大模型到 Agent 组成与最小循环\数据\demo-repo


## 工具不存在、参数错误、越界与空结果
这些情况不能都返回一句“出错了”。结构化错误帮助调用者决定修改参数、换工具、补充资料或停止。


In [2]:
repo_tools = RepoTools(DATA)
cases = [("invented_tool",{}), ("read_file",'{'), ("read_file",{"path":42}),
         ("read_file",{"path":"../private.txt"}), ("search_text",{"query":"not-a-real-symbol"})]
for name,args in cases:
    print(name,args,"=>",repo_tools.execute(name,args))


invented_tool {} => {'ok': False, 'error': {'code': 'unknown_tool', 'message': '工具未注册'}}
read_file { => {'ok': False, 'error': {'code': 'invalid_arguments', 'message': '请检查参数字段、类型和格式'}}
read_file {'path': 42} => {'ok': False, 'error': {'code': 'invalid_arguments', 'message': '请检查参数字段、类型和格式'}}
read_file {'path': '../private.txt'} => {'ok': False, 'error': {'code': 'out_of_scope', 'message': '只允许读取课程仓库已列出的文件'}}
search_text {'query': 'not-a-real-symbol'} => {'ok': True, 'data': {'query': 'not-a-real-symbol', 'hits': [], 'truncated': False}}


## 重复调用与预算耗尽
轮数上限保证实验结束，但不等于任务成功。此处每轮使用不同调用 ID，模拟模型反复读同一目录。重复 ID 是另一种协议错误，会单独停止。


In [3]:
repeating = ReplayModel([tool_response(f"repeat-{i}","list_files",{}) for i in range(5)])
limited = run_agent("查找不存在的入口",repeating,repo_tools,max_turns=3)
print("停止原因:",limited["status"],"工具次数:",limited["tool_calls"])
assert limited["status"]=="turn_budget"
empty = run_agent("test",ReplayModel([{"role":"assistant","content":""}]),repo_tools)
assert empty["status"]=="empty_answer"
print("空回答停止原因:",empty["status"])


停止原因: turn_budget 工具次数: 3
空回答停止原因: empty_answer


### 服务错误和工具错误在哪一层处理

配套课件：L02.07-S03、L02.08-S01。

同一个循环可以接不同模型适配器，前提是它们遵守同一份返回值与异常约定。下面让一个明确标注的测试适配器抛出课程 ModelError，检查循环能否保留停止原因；这不是向服务发送请求。

In [4]:
from agent_lab import ModelError as SharedModelError
def case_unavailable(messages, tools):
    raise SharedModelError("课堂注入：服务不可用，未发送网络请求")
case_failed_run = run_agent("读取配置", case_unavailable, RepoTools(DATA))
print("停止原因:", case_failed_run["status"], "工具次数:", case_failed_run["tool_calls"])
print("错误轨迹:", case_failed_run["trace"])
assert case_failed_run["status"] == "model_error"
assert case_failed_run["tool_calls"] == 0


停止原因: model_error 工具次数: 0
错误轨迹: [{'turn': 1, 'event': 'model_error', 'message': '课堂注入：服务不可用，未发送网络请求'}]


**结果解读**

模型调用失败时，循环应返回 model_error，并且工具次数为 0。未知工具或参数错误则属于工具观察，可能交给下一轮处理。异常类同名但不是同一个 Python 类，也可能导致捕获失败。

**小练习**

把异常改成一个新定义的同名 ModelError，为什么 except SharedModelError 捕获不到？

<details><summary>完成后展开参考分析</summary>

异常匹配依据类及继承关系，不依据类名字符串；适配器和循环应导入同一异常类型。不要用宽泛的 except 吞掉所有编程错误。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

### 练习：构造提前完成
让回放不使用任何工具就返回一个不存在的文件名。观察停止原因，解释为什么不能标记任务成功。然后将 max_tool_calls=1，观察成功轨迹为什么也会停止。


In [5]:
# TODO：构造只有最终回答的 ReplayModel，运行后检查 status、tool_calls、task_success。


In [6]:
early = run_agent("找入口",ReplayModel([{"role":"assistant","content":"入口是 loader.py。"}]),repo_tools)
print({k:early[k] for k in ["status","tool_calls","task_success"]})
assert early["status"]=="final_answer" and early["tool_calls"]==0
from agent_lab import success_replay
budgeted = run_agent("找入口",success_replay(),repo_tools,max_tool_calls=1)
assert budgeted["status"]=="tool_budget"
print("工具预算停止:",budgeted["status"])


{'status': 'final_answer', 'tool_calls': 0, 'task_success': 'requires_evidence_review'}
工具预算停止: tool_budget


## 接入 DeepSeek
在启动 Jupyter 的终端设置 `DEEPSEEK_API_KEY`，或使用课程平台提供的环境变量配置。不要把密钥写在 Notebook、输出或截图里。

`DEEPSEEK_MODEL` 可覆盖默认 `deepseek-flash`。模型名可能随服务升级而变化，真实调用前先读取当前账号的 `/models` 列表，再从实际返回值中选择。不要根据名称排序猜测哪个模型更新。

本课采用非思考模式、非流式请求，只读写工具协议所需的消息；不需要展示私有推理。

下面是完整 HTTP 接入代码。它不会自动重试收费请求，连接和响应长度异常会保留为失败状态。

来源：[DeepSeek 工具调用文档](https://api-docs.deepseek.com/guides/tool_calls/)、[对话接口](https://api-docs.deepseek.com/api/create-chat-completion/)。


In [7]:
# 与 run_agent 使用同一个异常类，确保模型错误能被循环捕获。
from agent_lab import ModelError


def list_deepseek_models(api_key=None):
    """读取当前账号实际可用的模型名，不根据版本字符串猜测。"""
    key = api_key or os.environ.get("DEEPSEEK_API_KEY", "")
    if not key:
        raise ModelError("未配置 DEEPSEEK_API_KEY；无法读取可用模型列表")
    request = urllib.request.Request(
        "https://api.deepseek.com/models",
        headers={"Authorization": "Bearer " + key})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            data = json.load(response)
        models = sorted(item["id"] for item in data["data"]
                        if isinstance(item, dict) and isinstance(item.get("id"), str))
        if not models:
            raise ModelError("账号没有返回可用模型，请核对权限或官方文档")
        return models
    except urllib.error.HTTPError as exc:
        raise ModelError(f"HTTP {exc.code}：无法读取模型列表，请核对鉴权或服务状态") from None
    except (urllib.error.URLError, TimeoutError, OSError):
        raise ModelError("读取模型列表失败或超时") from None
    except (KeyError, ValueError, TypeError):
        raise ModelError("模型列表响应格式不符合预期") from None


def validate_deepseek_model(model=None, api_key=None):
    """用当前账号的实际列表验证本次选择，不根据名称猜测新旧。"""
    selected = model or os.environ.get("DEEPSEEK_MODEL", "deepseek-flash")
    available = list_deepseek_models(api_key)
    if selected not in available:
        raise ModelError(
            f"所选模型 {selected!r} 不在当前账号的可用列表中；"
            f"当前可用模型为 {available}。请从列表中选择并更新 DEEPSEEK_MODEL")
    return selected, available


class DeepSeekModel:
    def __init__(self, model=None):
        self.key = os.environ.get("DEEPSEEK_API_KEY", "")
        self.model = model or os.environ.get("DEEPSEEK_MODEL", "deepseek-flash")
        if not self.key:
            raise ModelError("未配置 DEEPSEEK_API_KEY；本次不会发送请求")
        self.usage = []

    def __call__(self, messages, tools):
        payload = {"model": self.model, "messages": messages, "tools": tools,
                   "thinking": {"type": "disabled"}, "max_tokens": 1000, "stream": False}
        request = urllib.request.Request("https://api.deepseek.com/chat/completions",
                    data=json.dumps(payload).encode("utf-8"),
                    headers={"Authorization": "Bearer " + self.key, "Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                data = json.load(response)
            self.usage.append(data.get("usage", {}))
            choice = data["choices"][0]
            if choice.get("finish_reason") not in ("stop", "tool_calls"):
                raise ModelError("响应未正常完成：" + str(choice.get("finish_reason")))
            # 非思考模式只把正文和原样工具调用回传，不展示或依赖隐藏推理。
            message = choice["message"]
            return {k: message[k] for k in ("role", "content", "tool_calls") if k in message}
        except urllib.error.HTTPError as exc:
            raise ModelError(f"HTTP {exc.code}：请核对鉴权、模型名称、额度或服务状态") from None
        except (urllib.error.URLError, TimeoutError, OSError):
            raise ModelError("模型连接失败或超时；未自动重试") from None
        except (KeyError, ValueError, IndexError, TypeError):
            raise ModelError("模型响应格式不符合本课接口约定") from None


In [8]:
RUN_LIVE = False  # 需要真实调用时改为 True
selected_model = os.environ.get("DEEPSEEK_MODEL", "deepseek-flash")
print("已配置密钥:", bool(os.environ.get("DEEPSEEK_API_KEY")))
print("本次选择的模型:", selected_model)
if os.environ.get("DEEPSEEK_API_KEY"):
    available_models = list_deepseek_models()
    print("当前账号可用模型:", available_models)
    if selected_model not in available_models:
        print("当前选择已经不可用。请从上面的列表中选择，并设置 DEEPSEEK_MODEL。")
else:
    available_models = []
    print("配置密钥后才能读取当前账号的模型列表。")
print("本次允许真实调用:", RUN_LIVE)


已配置密钥: False
本次选择的模型: deepseek-flash
配置密钥后才能读取当前账号的模型列表。
本次允许真实调用: False


In [9]:
if RUN_LIVE:
    if selected_model not in available_models:
        raise ModelError("所选模型不在当前账号的可用列表中；请先更新 DEEPSEEK_MODEL")
    model = DeepSeekModel(selected_model)
    live = run_agent("找出配置加载入口。说明默认文件名、环境变量以及文件不存在时的行为，并给出文件行号。",
                     model,repo_tools,max_turns=6,max_tool_calls=10)
    live["model"] = model.model
    live["usage"] = model.usage
    live["mode"] = "real_deepseek"
    OUT = ROOT/".local/student-runs/l02"; OUT.mkdir(parents=True,exist_ok=True)
    (OUT/"deepseek-run.json").write_text(json.dumps(live,ensure_ascii=False,indent=2),encoding="utf-8")
    print("停止:",live["status"],"调用次数:",live["tool_calls"])
    print("回答:",live["answer"])
    print("用量（服务返回）:",live["usage"])
else:
    print("未执行真实 DeepSeek 请求。开启 RUN_LIVE 并配置环境变量后再验收。")


未执行真实 DeepSeek 请求。开启 RUN_LIVE 并配置环境变量后再验收。


## 智能体任务的人工验收
对照真实工具结果逐项检查，不按回答长度打分：

| 项目 | 本教学仓库的依据 |
|---|---|
| 定位实现 | config.py 的 load_settings，实际定义在第 7 行 |
| 找到调用关系 | main.py 第 2 行导入、第 6 行调用 |
| 环境变量与默认文件 | config.py 第 8 行 |
| 文件不存在时的行为 | config.py 第 9–10 行返回默认字典 |
| 证据来自执行 | trace 中有支持这些结论的成功工具结果 |

配置实际存在时，settings.json 的 timeout 是 15；缺失时默认 timeout 是 30。它们不能混为一谈。

如果没完成，标记失败并说明原因；不要手工补写轨迹来装成成功。后续“智能体评估”一节会把这些标准系统化。


In [10]:
# 用程序核对验收依据，防止文件修改后行号悄悄过时。
config = repo_tools.read_file("config.py")["lines"]
main = repo_tools.read_file("main.py")["lines"]
assert config[6]["text"] == "def load_settings():"
assert "STUDYBOX_CONFIG" in config[7]["text"]
assert "timeout" in config[9]["text"] and "30" in config[9]["text"]
assert "load_settings" in main[1]["text"] and "load_settings()" in main[5]["text"]
print("验收依据仍与文件一致；这不是对模型回答的自动语义评分。")


验收依据仍与文件一致；这不是对模型回答的自动语义评分。


## 实用任务：把运行记录变成验收报告
下面的检查器确认循环是否正常结束、关键文件是否真的读取、工具调用是否成功。它给出可追踪的结构化报告，但仍需要人工判断证据是否支持最终结论。


In [11]:
from agent_lab import success_replay

def review_run(candidate, required_files=("main.py", "config.py")):
    tool_events = [event for event in candidate["trace"] if event["event"] == "tool"]
    read_paths = []
    for event in tool_events:
        if event["name"] != "read_file":
            continue
        arguments = event["arguments"]
        if isinstance(arguments, str):
            arguments = json.loads(arguments)
        if isinstance(arguments, dict) and isinstance(arguments.get("path"), str):
            read_paths.append(arguments["path"])
    read_paths = sorted(set(read_paths))
    failed_calls = [event["id"] for event in tool_events if not event["result"]["ok"]]
    checks = {
        "finished_with_answer": candidate["status"] == "final_answer",
        "all_tool_calls_succeeded": not failed_calls,
        "required_files_read": set(required_files) <= set(read_paths),
        "answer_is_nonempty": bool((candidate.get("answer") or "").strip()),
    }
    return {"passed": all(checks.values()), "checks": checks,
            "read_paths": read_paths, "failed_call_ids": failed_calls,
            "tool_calls": candidate["tool_calls"]}

candidate = live if RUN_LIVE else run_agent(
    "找出配置加载入口，并说明缺少配置文件时的行为。",
    success_replay(), repo_tools)
report = review_run(candidate)
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["passed"]

OUT = ROOT/".local/student-runs/l02"; OUT.mkdir(parents=True, exist_ok=True)
(OUT/"run-review.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")


{
  "passed": true,
  "checks": {
    "finished_with_answer": true,
    "all_tool_calls_succeeded": true,
    "required_files_read": true,
    "answer_is_nonempty": true
  },
  "read_paths": [
    "config.py",
    "main.py"
  ],
  "failed_call_ids": [],
  "tool_calls": 3
}


273

## 提交
提交真实 DeepSeek 的成功或失败记录、一次无密钥失败注入记录、你的循环关键代码和人工验收表。

未配置服务的同学先完成 02A、02B 和这个 Notebook 的离线部分；真实 Agent 能力验收仍待补交。不要把回放视为最终模型能力成绩。

思考：当仓库从 4 个文件变成 4000 个文件，list_files 和整文件读取会遇到什么问题？下一节讲检索，第 4 节单独讲上下文，第 6 节用 Harness 汇总运行控制。
